<td>
<a href="https://colab.research.google.com/github/raoulg/MADS-DAV/blob/main/notebooks/lesson6/06.3-vectorspaces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
</td>


# 6.3 Vector spaces — what an organised one looks like

6.1 projected raw *columns* — pixels, measurements — into two dimensions. 6.2 built a
vector by hand, one trigram count at a time. This notebook looks at a vector nobody in
this room designed: the output of an encoder trained on a few million photos, published
once and downloaded rather than recomputed.

The question is not "does dimensionality reduction work on this" — 6.1 answered that.
It is: **what does a representation look like when it is actually good**, and how do you
check that a picture of it is showing you something real. Two ideas carry the answer —
cosine similarity, and a baseline computed without any picture at all — and both come
back in 6.4, where the vectors are yours and the data is much less cooperative.

> Needs `torch` and `vectormesh`: `uv sync --extra huggingface` once, locally.


In [ ]:
import numpy as np
import torch
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

from goad_toolkit.visualizer import PlotSettings, ProjectionPlot
from vectormesh import VectorCache


## A cache is vectors plus a manifest

Running an image through an encoder is the expensive part of this pipeline, and it does
not depend on what you plan to do with the result — so it gets done once, and published.
`VectorCache.from_hub` downloads that result: a folder of pre-computed vectors and a
`metadata.json` describing them, in place of a model download and a GPU.

This is the same idea the ML course's cache lesson opens with; the two lines below are
the whole cost of getting to a real, fine-grained, already-encoded dataset.


In [ ]:
tr = VectorCache.from_hub("pttrn-io/oxford-flowers-dinov2-small", split="train")
te = VectorCache.from_hub("pttrn-io/oxford-flowers-dinov2-small", split="test")

Xtr, ytr = tr["embed"][:], tr["label"][:]
Xte, yte = te["embed"][:], te["label"][:]
print(f"{len(tr):,} train vectors, {len(te):,} test vectors, {Xtr.shape[1]} dimensions each")
print(f"{te.features['label'].num_classes} flower species")


No pixels anywhere in that folder — 102 species of flower, reduced to a 384-number
fingerprint per photo by `facebook/dinov2-small`, downloaded in under a second. The
labels in this particular cache are class numbers rather than species names; that is
fine, because nothing below needs to know which number is a rose.


## An organised space, before anything is trained

If the encoder did its job, photos of the same species should already sit near each
other — not because anything has been fitted to this task, but because "near" in this
space is supposed to mean "similar". Eight species, projected to 2D with the same PCA
6.1 used, is the direct way to check that.


In [ ]:
flower_names = te.features["label"].names
rng = torch.Generator().manual_seed(0)
sample_classes = torch.randperm(len(flower_names), generator=rng)[:8]

mask_te = torch.isin(yte, sample_classes)
# torch ships PCA already — pca_lowrank centres X internally, U * S is the
# projection onto the top components, no new dependency for a 2D scatter.
U, S, _ = torch.pca_lowrank(Xte[mask_te], q=2)
coords = (U * S).numpy()
labels = np.array([f"class {i}" for i in yte[mask_te].numpy()])

settings = PlotSettings(
    figsize=(7, 6),
    title="Eight flower classes, PCA of raw DINOv2 vectors — nothing trained",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=coords, labels=labels, palette="tab10")


Eight tight, mostly separated clusters, from a *linear* projection of raw encoder
output. Compare this with 6.1's MNIST picture: two components of raw pixels there
carried 17% of the variance and produced a smear. Here the same two-component PCA
produces clusters, because the 384 numbers were already organised by species before
PCA ever touched them — the projection is not creating the structure, only letting you
see it.


## Cosine similarity, properly explained

Every distance used so far — Manhattan in 6.2, Euclidean implicitly in every PCA plot —
treats a vector's length as part of what it is measuring. Cosine similarity does not: it
compares *direction* only, which makes it the right metric whenever length is not part
of the meaning.

A small example makes the difference concrete. Three 2D vectors, standing in for three
short pieces of text:


In [ ]:
a = np.array([1.0, 2.0])   # "a message"
b = np.array([3.0, 6.0])    # "a message a message a message" -- same content, said three times
c = np.array([2.0, 1.0])    # a different message entirely


def euclidean(u, v):
    return np.linalg.norm(u - v)


def cosine(u, v):
    return (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))


print(f"euclidean(a, b) = {euclidean(a, b):.2f}   cosine(a, b) = {cosine(a, b):.2f}")
print(f"euclidean(a, c) = {euclidean(a, c):.2f}   cosine(a, c) = {cosine(a, c):.2f}")


By Euclidean distance, `a` sits closer to `c` — a genuinely different message — than
to `b`, which is `a` repeated three times and says nothing new. By cosine, `b` is
identical to `a` (`1.00`, same direction) and `c` is clearly different (`0.80`). Cosine
is the metric that agrees with what actually happened: repeating yourself is not saying
something new, and length should not be allowed to look like it is.

This is leerdoel 6.6's metric, and it is why cosine — not Euclidean, not Manhattan —
is the standard choice for comparing sentence and document embeddings: a three-word
message and a three-paragraph one can point the same direction, and cosine is the only
one of the three that would say so.


### Does it matter here?

The property only *bites* when length varies for reasons that have nothing to do with
meaning — a short message against a long one. Check whether that is even true of these
flower vectors before assuming cosine will beat the alternatives on this particular
dataset.


In [ ]:
norms = Xtr.norm(dim=1)
print(f"vector length: mean {norms.mean():.1f}, std {norms.std():.2f} "
      f"({norms.std() / norms.mean():.1%} of the mean)")


Under 2% variation. `facebook/dinov2-small` normalises internally, so almost none of
the information in these particular vectors lives in their length — which means, on
*this* dataset, cosine, Euclidean and Manhattan are about to agree, not because the
choice does not matter in general, but because this encoder already did the equalising
that cosine would otherwise have to do. 6.4 is a different story: nobody normalises a
WhatsApp message for you, and a three-word "haha" has nowhere near the length of a
three-paragraph story.


## The baseline: 1-nearest-neighbour, no training at all

Normalise, take a dot product, find the closest training vector, copy its label. Four
lines, no loss function, no optimiser — and it is the number every embedding-based
method in this space has to beat.


In [ ]:
def one_nn_accuracy(Xtr, ytr, Xte, yte, p=None):  # noqa: N803
    if p is None:  # cosine: normalise first, then it is just a dot product
        train = torch.nn.functional.normalize(Xtr, dim=1)
        test = torch.nn.functional.normalize(Xte, dim=1)
        nearest = (test @ train.T).argmax(dim=1)
    else:
        nearest = torch.cdist(Xte, Xtr, p=p).argmin(dim=1)
    return (ytr[nearest] == yte).float().mean().item()


print(f"cosine     : {one_nn_accuracy(Xtr, ytr, Xte, yte):.1%}")
print(f"euclidean  : {one_nn_accuracy(Xtr, ytr, Xte, yte, p=2):.1%}")
print(f"manhattan  : {one_nn_accuracy(Xtr, ytr, Xte, yte, p=1):.1%}")


**99.1% on 102 fine-grained classes, with nothing that could be called a model** — and
the three metrics land within two tenths of a point of each other, exactly as the length
check above predicted. This is the baseline's real job: it answers *"is the clustering
in the picture above real, or is it something the projection invented?"* — computed here
in the original 384-dim space, not the 2D one, so it verifies what that picture only
suggested.


## Checking it the way 6.1 taught you to

A single perplexity, once, is a demo. 6.1's rule was to run more than one and see what
holds — membership tends to survive, geometry does not — and to check the picture
against a number computed without a picture at all. Same habit, this dataset.


In [ ]:
PERPLEXITIES = (5, 30, 100)
mask_tr = torch.isin(ytr, sample_classes)
X = Xtr[mask_tr].numpy()
y = np.array([f"class {i}" for i in ytr[mask_tr].numpy()])

embeddings = {
    p: TSNE(n_components=2, perplexity=p, random_state=42, init="pca").fit_transform(X)
    for p in PERPLEXITIES
}

settings = PlotSettings(
    figsize=(11, 4),
    title="The same eight classes, three perplexities",
    xlabel="",
    ylabel="",
    max_cols=3,
    subplot_titles=[f"perplexity {p}" for p in PERPLEXITIES],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=len(PERPLEXITIES))
for ax, p in zip(axes, PERPLEXITIES):
    host.plot_on_axes(ProjectionPlot(settings), ax, coordinates=embeddings[p],
                      labels=y, palette="tab10", legend=False, s=8)

original_space_silhouette = silhouette_score(X, y)
print(f"\nsilhouette, original 384-dim space: {original_space_silhouette:.3f}")
for p, embedding in embeddings.items():
    print(f"silhouette, t-SNE at perplexity {p:>3d}: {silhouette_score(embedding, y):.3f}")

Membership holds at every perplexity — the eight classes stay eight classes from 5 to
100. The 2D silhouette scores (0.71–0.77) run higher than the 384-dim one (0.41), which
is expected and not a contradiction: t-SNE optimises exactly for tight, separated 2D
neighbourhoods, so its own score is not directly comparable to the original space's. What
*is* comparable, and what actually matters, is that the ranking and the story do not
change with the knob — which is the check 6.1 spent a whole section arguing you should
never skip.


## What to carry into 6.4

1. **A cache is vectors plus a manifest.** Read `metadata.json` — or here, the shapes and
   the label count — before writing a line of model code.
2. **Cosine ignores magnitude on purpose.** Check whether that purpose is even needed on
   your data — it was not, here, because the encoder had already equalised lengths. It
   will be needed the moment length varies for reasons that have nothing to do with
   meaning.
3. **A baseline computed without a picture is what tells you the picture is real.** 99.1%
   in 384 dimensions is the number the 2D scatter only hinted at.

Everything above worked because one row was one flower, cleanly labelled, encoded once.
6.4 removes that luxury: the vectors are yours, a row is a WhatsApp message, and a large
share of those are five words or fewer. The first problem there is not which metric to
use — it is what to even call "one row".
